In [37]:
import ccxt
import pandas as pd
import numpy as np
import datetime

In [38]:
exchange = ccxt.binance()

def fetchOHLCV(symbol, timeframe, since):
	df = exchange.fetch_ohlcv(symbol=f'{symbol}', timeframe=timeframe, since=int(datetime.datetime.timestamp(since) * 1000), limit=365)
	df = pd.DataFrame(df, columns=['timestamp', 'Open', 'High', 'Low', 'Close', 'Volume'])
	df['Datetime'] = pd.to_datetime(df['timestamp'] * 1000 * 1000, utc=True).dt.tz_convert('Australia/Adelaide')
	df = df.set_index('Datetime')
	return df

start = datetime.datetime.strptime('2021-01-01', '%Y-%m-%d')

df = fetchOHLCV('BTC/USDT', '1d', start)
df

,timestamp,Open,High,Low,Close,Volume
Datetime,,,,,,
2021-01-01 10:30:00+10:30,1609459200000,28923.63,29600.00,28624.57,29331.69,54182.925011
2021-01-02 10:30:00+10:30,1609545600000,29331.70,33300.00,28946.53,32178.33,129993.873362
2021-01-03 10:30:00+10:30,1609632000000,32176.45,34778.11,31962.99,33000.05,120957.566750
2021-01-04 10:30:00+10:30,1609718400000,33000.05,33600.00,28130.00,31988.71,140899.885690
2021-01-05 10:30:00+10:30,1609804800000,31989.75,34360.00,29900.00,33949.53,116049.997038
...,...,...,...,...,...,...
2021-12-27 10:30:00+10:30,1640563200000,50775.48,52088.00,50449.00,50701.44,28792.215660
2021-12-28 10:30:00+10:30,1640649600000,50701.44,50704.05,47313.01,47543.74,45853.339240
2021-12-29 10:30:00+10:30,1640736000000,47543.74,48139.08,46096.99,46464.66,39498.870000


In [39]:
o = df['Open'].to_numpy()
h = df['High'].to_numpy()
l = df['Low'].to_numpy()
c = df['Close'].to_numpy()

In [40]:
from double7 import Double7

bt = Double7(7)
for i in range(len(h)):
    bt.update(i, df.index, o, h, l, c)

pd.DataFrame(bt.records)

,entry_index,entry_price,entry_timestamp,exit_index,exit_price,exit_timestamp,percentage_change
0,7,39432.48,2021-01-08 10:30:00+10:30,20,35468.23,2021-01-21 10:30:00+10:30,-10.053261
1,29,34246.28,2021-01-30 10:30:00+10:30,54,48891.00,2021-02-24 10:30:00+10:30,42.762951
2,62,50349.37,2021-03-04 10:30:00+10:30,82,54342.80,2021-03-24 10:30:00+10:30,7.931440
3,88,57635.46,2021-03-30 10:30:00+10:30,97,55953.44,2021-04-08 09:30:00+09:30,-2.918377
4,100,59769.13,2021-04-11 09:30:00+09:30,108,56150.01,2021-04-19 09:30:00+09:30,-6.055166
5,118,54846.23,2021-04-29 09:30:00+09:30,132,49537.15,2021-05-13 09:30:00+09:30,-9.679936
6,154,39246.78,2021-06-04 09:30:00+09:30,158,33556.96,2021-06-08 09:30:00+09:30,-14.497546
7,164,39020.56,2021-06-14 09:30:00+09:30,171,35600.17,2021-06-21 09:30:00+09:30,-8.765610
8,180,35911.72,2021-06-30 09:30:00+09:30,189,32875.71,2021-07-09 09:30:00+09:30,-8.454092
9,204,33634.10,2021-07-24 09:30:00+09:30,215,38207.04,2021-08-04 09:30:00+09:30,13.596142


In [41]:
import plotly.graph_objects as go

records_df = pd.DataFrame(bt.records)

# Create figure
fig = go.Figure()

# Add candlestick chart
fig.add_trace(go.Candlestick(
    x=df.index,
    open=df["Open"],
    high=df["High"],
    low=df["Low"],
    close=df["Close"],
    name="Candlestick"
))

# Add buy signals (green arrows)
fig.add_trace(go.Scatter(
    x=records_df["entry_timestamp"],
    y=records_df["entry_price"] * 0.9,
    mode="markers",
    marker=dict(symbol="triangle-up", color="green", size=10),
    name="Buy Signal"
))

# Add sell signals (red arrows)
fig.add_trace(go.Scatter(
    x=records_df["exit_timestamp"],
    y=records_df["exit_price"] * 1.1,
    mode="markers",
    marker=dict(symbol="triangle-down", color="red", size=10),
    name="Sell Signal"
))

# Update layout
fig.update_layout(
    title="Candlestick Chart with Buy and Sell Signals",
    xaxis_title="Datetime",
    yaxis_title="Price",
    xaxis_rangeslider_visible=True,
    height=800
)

# Show figure
fig.show()

In [42]:
from tabulate import tabulate

total_trade = records_df.shape[0]
winners = (records_df['percentage_change'] > 0).sum()
losers = (records_df['percentage_change'] <= 0).sum()
win_ratio = round(winners/total_trade * 100, 2) if total_trade else 0
pnl = (records_df['percentage_change'] + 100) / 100
cumulative_return = pnl.prod() * 100 - 100
total_profit = records_df[records_df['percentage_change'] > 0]['percentage_change'].sum()
total_loss = records_df[records_df['percentage_change'] <= 0]['percentage_change'].sum()
avg_profit_per_trade = round(total_profit/winners, 2) if winners else 0
avg_loss_per_trade = round(total_loss/losers, 2) if losers else 0
avg_pnl_per_trade = round(cumulative_return/total_trade, 2) if total_trade else 0
risk_reward = f"1:{round(abs(avg_profit_per_trade/avg_loss_per_trade), 2)}" if avg_loss_per_trade else "N/A"

data = [
            ('Total Trade', total_trade),
            ('Cumulative Return', cumulative_return),
            ('Winners', winners),
            ('Losers', losers),
            ('% Win Ratio', win_ratio),
            ('% Average Profit per Trade', avg_profit_per_trade),
            ('% Average Loss per Trade', avg_loss_per_trade),
            ('% Average PNL per Trade', avg_pnl_per_trade),
            ('Risk Reward', risk_reward)
        ]

print(tabulate(data, headers=['Parameters', 'Values'], tablefmt='psql'))

+----------------------------+--------------------+
| Parameters                 | Values             |
|----------------------------+--------------------|
| Total Trade                | 17                 |
| Cumulative Return          | -4.017068134573478 |
| Winners                    | 5                  |
| Losers                     | 12                 |
| % Win Ratio                | 29.41              |
| % Average Profit per Trade | 19.97              |
| % Average Loss per Trade   | -7.34              |
| % Average PNL per Trade    | -0.24              |
| Risk Reward                | 1:2.72             |
+----------------------------+--------------------+
